# Data Audit & Schema Review

**Name**: May Bui

**Date**: 2026-09-14

**Description**: <br>Run on VSCode
Verify benchmark integrity, document corpus field layout, log data-quality issues (rule-format mismatch, corpus-missing rules, exists label inconsistency)

In [ ]:
import json, glob, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data

In [ ]:
# Change file path as needed to match your local environment
docs = glob.glob("D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/benchmark/documents/*.docx")
answer_key = json.load(open("D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/benchmark/answer-key.json"))
manifest = json.load(open("D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/benchmark/manifest.json"))

## 1. File check
Verify that all files documented in the data dictionary are present and confirm that the number of files and records matches the expected counts.

In [ ]:
print("answer-key record keys:", sorted(answer_key[0].keys()))
print("citation entry keys:", sorted(answer_key[0]["citations"][0].keys()))
print("manifest keys:", sorted(manifest.keys()) if isinstance(manifest, dict) else sorted(manifest[0].keys()))


## 2. Schema Validation

Check that each file follows the documented schema, including expected fields, data types, column names, and overall structure.

In [ ]:
print("docs:", len(docs))
print("answer key:", len(answer_key))

corpus_info = {}
for name in ["statutes", "rules", "cases", "rules-evidence"]:
    with open(f"D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/corpus/{name}.jsonl", encoding="utf-8") as f:
        lines = f.readlines()
    corpus_info[name] = {"count": len(lines), "fields": sorted(json.loads(lines[0]).keys())}

for name, info in corpus_info.items():
    print(name, info["count"], info["fields"])

## 3. Data Consistency and Distribution
Review summary statistics and value distributions to identify unexpected patterns. This includes checking numerical ranges, missing values, category frequencies, dataset splits, and potential outliers.

In [ ]:
df = pd.json_normalize(answer_key)
print(df["condition"].value_counts(), "\n")
print(df["doc_type"].value_counts(), "\n")

citations = df[["doc_id", "condition", "citations"]].explode("citations")
citations = pd.concat(
    [citations.drop(columns="citations"), citations["citations"].apply(pd.Series)],
    axis=1
)
print(citations["type"].value_counts(), "\n")
print(citations["exists"].value_counts(), "\n")
print(citations[~citations["exists"]].groupby("condition").size())  # exists:false = broken citations by condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

df["condition"].value_counts().plot(kind="bar", ax=axes[0, 0], color="#4C72B0", rot=0)
axes[0, 0].set_title("Documents by condition")

df["doc_type"].value_counts().plot(kind="barh", ax=axes[0, 1], color="#4C72B0")
axes[0, 1].set_title("Documents by doc_type")
axes[0, 1].invert_yaxis()

citations["type"].value_counts().plot(kind="bar", ax=axes[1, 0], color="#4C72B0", rot=0)
axes[1, 0].set_title("Citations by type")

citations[~citations["exists"]].groupby("condition").size().plot(
    kind="bar", ax=axes[1, 1], color="#C44E52", rot=0
)
axes[1, 1].set_title("exists:false citations by condition\n(clean should be 0)")

plt.tight_layout()
plt.show()

When I group citation that has `exists: false` by document condition, the counts are `clean: 21, corrupt: 289` when `clean` should show 0, since "clean" is defined as having no fabricated citations.

Let's inspect the 21 clean records to see if they are fabricated citations or an answer-key labeling error.

## Referential Integrity Check
Validate relationships across files by confirming that reference fields, such as citation identifiers or doc_id values, correctly map to existing records in the corresponding datasets.

In [ ]:
clean_false = citations[(citations["condition"] == "clean") & (~citations["exists"])]
print(len(clean_false))
print(clean_false[["doc_id", "cite", "type", "injected"]].to_string())

* 7 statute cites (AS 18.66.990 ×4, AS 18.66.180 ×2, AS 11.56.807 ×1)
* 14 case cites with `injected: False` 

-> These are probably labeling anomalies

## Edge Case and Format Inspection
Inspect a sample of records to identify formatting inconsistencies, unusual cases, or deviations from the documented specification that may not be captured through automated checks.

In [ ]:
statutes_corpus = {json.loads(l)["citation"] for l in open("D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/corpus/statutes.jsonl", encoding="utf-8")}
cases_corpus = [json.loads(l) for l in open("D:/.BTT/Legal1A_FallAIStudio/Legal-1A-legal-ai-accuracy-evaluation-red-teaming-and-improvement-recommendations/data/corpus/cases.jsonl", encoding="utf-8")]

In [ ]:
for cite in clean_false.loc[clean_false["type"] == "statute", "cite"]:
    print(cite, "->", "FOUND" if cite in statutes_corpus else "NOT FOUND")

reporter_lookup = {c["reporter_cite"]: c for c in cases_corpus}

print("\n")
def extract_reporter(cite):
    m = re.search(r"\d+ P\.\d[a-z]* \d+", cite)
    return m.group(0) if m else None

for cite in clean_false.loc[clean_false["type"] == "case", "cite"]:
    r = extract_reporter(cite)
    match = reporter_lookup.get(r)
    print(cite, "->", r, "FOUND as:", match["case_name"] if match else "NOT FOUND")

* 20 of 21 resolve in the corpus. Only `AS 11.56.807` is missing. 
* Statutes (6/7 found): string matches, no normalization needed, just a mislabeled answer key.
* Cases (all found 14/14): the mismatch comes from leading prose stitched onto the citation and from party-name.
* As for `AS 11.56.807` -> Confirm in `Title 11.56` has no matching section in statutes.jsonl

## Summary and Findings

### Inventory

| Component | Expected (per DATA_DICTIONARY.md §2–3) | Found |
|---|---|---|
| Documents | 200 | 200 ✅ |
| Answer-key records | 200 | 200 ✅ |
| `statutes.jsonl` | ~500 | 500 ✅ |
| `rules.jsonl` | ~155 | 155 ✅ |
| `rules-evidence.jsonl` | ~302 | 302 ✅ |
| `cases.jsonl` | ~1,255 | 1,255 ✅ |

### Schema

- Answer-key record keys, citation-entry keys, and manifest keys match `DATA_DICTIONARY.md` §4.1–§4.2 exactly.
- Corpus schemas match §3: `statutes`/`rules`/`cases` expose `citation` + `source_text`; `rules-evidence` differs as documented, using `ruleNumber`/`text` instead.
- Two minor naming quirks noticed in the corpus field lists: `case_type` appears on `statutes.jsonl`/`rules.jsonl` records (unexpected on non-case records — likely a leftover shared-template field), and `familySalient` in `rules-evidence.jsonl` is the only camelCase field name where everything else is snake_case, consistent with that file being built by a different process.

### Distributions

- Condition split: 103 corrupt / 97 clean.
- doc_type split: 42 / 42 / 42 / 42 across four types, 32 for `married_divorcing_with_children`.
- Citation types: 1,546 statute / 1,172 case / 772 rule (3,490 total).
- `exists`: 3,180 true / 310 false.

### Findings

1. **Answer-key `exists` labels are not fully trustworthy (critical).** 310 citations are marked `exists: false`; 21 of those sit in `clean` documents, which by definition should contain zero fabricated citations. All 21 have `injected: False`, confirming these are not deliberately planted errors but a labeling issue.

2. **20 of the 21 anomalous citations genuinely exist in the corpus.** 6 of 7 statute cites match the corpus exactly (`AS 18.66.990`, `AS 18.66.180`); all 14 case cites resolve once matched by reporter citation instead of the literal string — the mismatch is caused by leading prose stitched onto the citation text (e.g. "Alaska Supreme Court. See ...", "Petitioner. In ...") and by party-name drift (e.g. `Bromley` vs. the corpus's `State, Child Support Enforcement Division v. Bromley`).

3. **Only `AS 11.56.807` is genuinely fabricated and unmarked** — no Title 11.56 section exists in `statutes.jsonl` at all.

4. **Implication for the detector.** A strict existence checker built to match the literal `cite` string will disagree with the answer key on ~20 labels before it's even built wrong on purpose. Do not treat `exists` as ground truth without first normalizing case citations (strip leading prose, match by reporter). Log this as a known data limitation and report both "score vs. key" and "score vs. corrected ground truth" once the detector is built.